# Idea 4 · Lesson 3 — From a PPI complex to two residue graphs

The research question (see the repo README and the Obsidian research vault) is whether the **consensus** of three explainability methods predicts experimentally-measured binding hotspots better than any single method. This first notebook builds the data primitive everything else stands on: turning a two-chain protein complex into a **pair of graphs**, exactly as Sendin (2025) did on PINDER.

We use **barnase–barstar (PDB `1BRS`)** as the running example — the textbook alanine-scanning hotspot system, and one with rich `SKEMPI v2.0` ΔΔG data we'll validate against in lesson 7.

> **Run order matters.** These cells build on each other — run them top to bottom (*Run All*). Heavy steps (training, Integrated Gradients) are small by design but still compute; on a shared machine, run them when the box is idle.

## Setup

Locate the shared helpers and the data cache.

In [ ]:
import os, sys
HERE = os.path.abspath("")
ROOT = HERE if os.path.isfile(os.path.join(HERE, "ig", "idea4_common.py")) \
    else os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "ig"))
DATA = os.path.join(ROOT, "data")
WEIGHTS = os.path.join(DATA, "idea4_struct2graph.pt")
print("repo root:", ROOT)

In [ ]:
from idea4_common import (fetch_pdb, parse_chain, contact_graph,
                          interface_mask, get_device)
import numpy as np
import matplotlib.pyplot as plt
print('device:', get_device())

## Step 1 — Download the complex

`fetch_pdb` pulls the structure from RCSB once and caches it. It pins `certifi`'s CA bundle so the download works inside the venv even when a conda base environment has exported a stale `SSL_CERT_FILE`.

In [ ]:
pdb_path = fetch_pdb('1brs', cache_dir=DATA)
print('cached at:', pdb_path)

## Step 2 — Parse the two chains

In `1BRS`, chain **A** is barnase (the enzyme / receptor) and chain **D** is barstar (its inhibitor / ligand). `parse_chain` keeps only standard residues with a Cα atom and returns the sequence, the Cα coordinates, and the author residue numbers (we need those numbers in lesson 7 to line residues up with SKEMPI mutations).

In [ ]:
seq_a, ca_a, resnums_a = parse_chain(pdb_path, 'A')
seq_d, ca_d, resnums_d = parse_chain(pdb_path, 'D')
print(f'chain A (barnase): {len(seq_a)} residues')
print(f'chain D (barstar): {len(seq_d)} residues')
print('barnase seq:', seq_a)

## Step 3 — Build a residue contact graph per chain

Each residue becomes a node (positioned at its Cα); an edge joins any two residues whose Cα atoms are within **8 Å** — the standard contact cutoff. Node features are one-hot amino acids (swap in ESM-2 embeddings later for the multi-modal extension Sendin flagged).

In [ ]:
g_a = contact_graph(seq_a, ca_a, threshold=8.0)
g_d = contact_graph(seq_d, ca_d, threshold=8.0)
print('barnase graph:', g_a.num_nodes, 'nodes /', g_a.num_edges, 'edges')
print('barstar graph:', g_d.num_nodes, 'nodes /', g_d.num_edges, 'edges')
print('node feature shape:', tuple(g_a.x.shape))

## Step 4 — Sanity-check the graph as a contact map

A contact map is just the graph's adjacency matrix drawn as an image. The diagonal band is the protein backbone (residue *i* near *i±1*); off-diagonal blocks are tertiary contacts where the chain folds back on itself.

In [ ]:
A = np.zeros((g_a.num_nodes, g_a.num_nodes))
ei = g_a.edge_index.numpy()
A[ei[0], ei[1]] = 1
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(A, cmap='Greys', origin='lower')
ax[0].set_title('Barnase contact map (8 Å)')
ax[0].set_xlabel('residue'); ax[0].set_ylabel('residue')
deg = A.sum(1)
ax[1].bar(range(len(deg)), deg, color='steelblue')
ax[1].set_title('Per-residue contact degree')
ax[1].set_xlabel('residue'); ax[1].set_ylabel('# contacts')
plt.tight_layout(); plt.show()

## Step 5 — Where is the interface?

`interface_mask` flags residues on each chain whose Cα lies within a cutoff of *any* Cα on the partner chain — a Cα approximation of the binding interface. These are the residues most likely to contain hotspots, and the structural reference we'll compare the saliency rankings against (with SKEMPI as the experimental ground truth).

In [ ]:
mask_a, mask_d = interface_mask(ca_a, ca_d, cutoff=8.0)
print(f'barnase interface residues: {int(mask_a.sum())} / {len(mask_a)}')
print(f'barstar interface residues: {int(mask_d.sum())} / {len(mask_d)}')
iface_resnums = [resnums_a[i] for i in np.where(mask_a)[0]]
print('barnase interface residue numbers:', iface_resnums)

In [ ]:
plt.figure(figsize=(11, 2.4))
plt.bar(range(len(mask_a)), mask_a.astype(int), color='crimson', width=1.0)
plt.title('Barnase: interface residues (1 = within 8 Å of barstar)')
plt.xlabel('residue index'); plt.yticks([0, 1])
plt.tight_layout(); plt.show()

## Recap

We turned one PPI complex into two PyG graphs and located its interface — all from a single cached `.pdb`. **Lesson 4** trains a Struct2Graph classifier (shared-weight GCN + mutual attention) on a small set of such complexes; **lessons 5–7** explain it three ways and test the consensus against SKEMPI hotspots.